# Zobépine - consolidation des chroniques

Une seule sonde : **CTD** (Diver autonome : niveau, conductivité, température).
Pas de choix de sonde, pas de fusion, pas de points de contrôle.

Les fonctions communes sont dans la librairie `ouysse`. Ce notebook ne garde que
ce qui est propre à la station : chemins, mesures écartées, réglages du filtre.

## 1. Imports

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ouysse
from ouysse import *          # fonctions communes a toutes les stations

print("ouysse-hydro", ouysse.__version__)

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Zobépine\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Pluie_BV_Ouysse.csv"

OLDDATA_PATH     = os.path.join(BASE, r"Données consolidées\Zobépine_Old.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, r"Données consolidées\Zobépine_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, r"Données consolidées\Zobépine_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, r"Données consolidées\Graphes.svg")

PREFIXE_CTD = "Zobépine"
BARO_COL    = "Patm Thémines [hPa]"
PAS         = "1h"

PARAMETRES = ["Niveau_(cm)", "Conductivité", "Température"]

## 3. CTD : lecture, UTC et compensation barométrique

`lire_CTD` encaisse les pièges du format Diver (en-tête à une ligne variable, pied
`END OF DATA`, virgules décimales, mS/cm ou µS/cm).

**Pas de table UTC pour cette station** : les horodatages sont repris
tels quels. Si les exports sont en heure locale, la chronique l'est aussi.
Pour corriger, ajouter `UTC_CTD.xlsx` et passer `utc` à True dans
`tests/stations_ctd.py`.

In [ ]:
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD = lire_CTD(nom, CTD_PATH, PAS)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Cond_(µS/cm)": "Conductivité", "Température[°C]": "Température"})
    morceaux.append(m[["Date/time"] + PARAMETRES])
    print(f"  {nom:45s} {len(CTD)} lignes")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s), {len(merge_ctd_df)} enregistrements.")

## 4. Raccordement à l'ancienne chronique

L'ancien fichier consolidé et les campagnes récentes sont la **même sonde CTD**,
séparées par un trou d'exploitation : le décalage est mesuré à la jonction et
appliqué aux campagnes, pour que la chronique soit continue.

In [ ]:
#: Les anciens consolidés n'ont pas tous les mêmes en-têtes : les clés absentes
#: sont ignorées, ce dictionnaire couvre les deux conventions rencontrées.
RENOMMAGE_OLD = {
    "Date/time": "DATE",
    "NIVEAU": "Niveau_(cm)",
    "CONDUCTIVITE": "Conductivité",
    "TEMPERATURE CTD": "Température",
    "Cond_(µS/cm)": "Conductivité",
    "Temp_(°C)": "Température",
}

olddata_df = pd.read_excel(OLDDATA_PATH).rename(columns=RENOMMAGE_OLD)
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce")
olddata_df = olddata_df.dropna(subset=["DATE"]).sort_values("DATE")

manquantes = [c for c in PARAMETRES if c not in olddata_df]
if manquantes:
    print(f"Absentes de l'ancien consolidé : {manquantes}\n"
          f"  colonnes lues : {list(olddata_df.columns)}\n"
          f"  compléter RENOMMAGE_OLD si l'une d'elles porte un autre nom")

merge_ctd_df = raccorder_campagnes(olddata_df, merge_ctd_df, [
    (nom, col, col, unite)
    for nom, col, unite in [("Niveau", "Niveau_(cm)", "cm"),
                            ("Conductivité", "Conductivité", "µS/cm")]
    if col in olddata_df])

## 5. Assemblage sur la grille horaire

In [ ]:
full_data = sur_grille([empiler([olddata_df, merge_ctd_df], PARAMETRES)], PAS)

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data)

full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nfichier fusionné : {SORTIE_CONSOLIDE}")

## 6. Corrections capteur

`VOIES_ECARTEES` met des mesures à l'écart. Il n'y a pas de sonde de secours ici :
la lacune reste, et l'interpolation ne comblera pas plus de 12 h.

In [ ]:
#: (début, fin, colonne, motif) : mesures mises à l'écart.
VOIES_ECARTEES = [
]

print("Voies écartées :")
full_data = ecarter(full_data, VOIES_ECARTEES)

## 7. Filtre IQR et lissage

In [ ]:
FENETRE_IQR, K_IQR = "24h", 0.1   # k = 0 : pas de filtre
LISSAGE_H = 6                     # 0 = pas de lissage ; sinon médiane glissante, en heures

avant = full_data["Conductivité"]
full_data["Conductivité"] = filtre_iqr(avant, FENETRE_IQR, K_IQR, lissage_h=LISSAGE_H)
full_data["Conductivité_Moyenne_Mobile"] = full_data["Conductivité"].rolling("6h", center=True).mean()

graphe([(avant, "avant IQR et lissage", "darkorange"),
        (full_data["Conductivité"], "après IQR et lissage", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)")

## 8. Cote NGF, interpolation et statuts

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur
est mesurée, interpolée ou manquante.

In [ ]:
NIVEAU_NGF = None       # cote du zéro de l'échelle, None si elle n'est pas connue
MAX_TROU_H = 12

full_data = interpoler_avec_statut(full_data, PARAMETRES, MAX_TROU_H, PAS)

if NIVEAU_NGF is not None:
    full_data["Niveau_(mNGF)"] = NIVEAU_NGF + full_data["Niveau_(cm)"] / 100
    full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
    print(f"Zéro de l'échelle à {NIVEAU_NGF:.4f} m NGF")
display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 9. Sauvegarde et graphe de synthèse

In [ ]:
finaux = [c for c in PARAMETRES + ["Niveau_(mNGF)"] if c in full_data]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}") if c in full_data]
sortie = full_data[colonnes].copy()
sortie.attrs["ouysse"] = ouysse.__version__
sortie.to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(sortie)} pas x {len(colonnes)} colonnes "
      f"(ouysse-hydro {ouysse.__version__})")

graphe_synthese(full_data, "Niveau_(cm)", "Niveau (cm)", pluie=PLUIE_PATH, sortie=SORTIE_SVG)

In [ ]:
graphe_statuts(full_data, finaux)